In [1]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd

SNAPSHOT_DATE = "2026-02-03"  # cámbialo cuando vuelvas a correr
DB_NAME = "AdventureWorks2019"

DATA_OUT = Path("data/out")
DATA_OUT.mkdir(parents=True, exist_ok=True)

# Ajusta estas 2 si cambian
SQL_HOST = os.getenv("SQL_HOST", "localhost")
SQL_PORT = os.getenv("SQL_PORT", "1433")

SQL_USER = os.getenv("SQL_USER", "sa")
SQL_PASSWORD = os.getenv("SQL_PASSWORD", "StrongPass123!")  # <-- pon el real

In [2]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# ODBC Driver 18 + TrustServerCertificate evita el tema SSL self-signed
odbc_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={SQL_HOST},{SQL_PORT};"
    f"DATABASE={DB_NAME};"
    f"UID={SQL_USER};PWD={SQL_PASSWORD};"
    "Encrypt=yes;TrustServerCertificate=yes;"
    "Connection Timeout=30;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(odbc_str)}")

# sanity check
pd.read_sql("SELECT DB_NAME() AS db;", engine)

,db
0,AdventureWorks2019


In [3]:
snapshot_sql = """
WITH maxd AS (
  SELECT MAX(OrderDate) AS max_order_date
  FROM Sales.SalesOrderHeader
),
sales_12m AS (
  SELECT
      sod.ProductID AS sku_id,
      SUM(sod.LineTotal) AS revenue_12m,
      SUM(sod.OrderQty)  AS sales_units_12m
  FROM Sales.SalesOrderDetail sod
  JOIN Sales.SalesOrderHeader soh
      ON sod.SalesOrderID = soh.SalesOrderID
  CROSS JOIN maxd
  WHERE soh.OrderDate >= DATEADD(year, -1, maxd.max_order_date)
  GROUP BY sod.ProductID
),
cost_latest AS (
  SELECT
      pch.ProductID AS sku_id,
      pch.StandardCost AS unit_cost,
      ROW_NUMBER() OVER (
          PARTITION BY pch.ProductID
          ORDER BY pch.StartDate DESC
      ) AS rn
  FROM Production.ProductCostHistory pch
),
inv AS (
  SELECT
      pi.ProductID AS sku_id,
      SUM(pi.Quantity) AS inventory_units
  FROM Production.ProductInventory pi
  GROUP BY pi.ProductID
)
SELECT
    p.ProductID AS sku_id,
    p.Name      AS sku_name,
    c.Name      AS category,
    sc.Name     AS subcategory,

    COALESCE(inv.inventory_units, 0) AS inventory_units,
    COALESCE(cost.unit_cost, 0)      AS unit_cost,

    COALESCE(sales_12m.sales_units_12m, 0) AS sales_units_12m,
    COALESCE(sales_12m.revenue_12m, 0)     AS revenue_12m

FROM Production.Product p
LEFT JOIN Production.ProductSubcategory sc
    ON p.ProductSubcategoryID = sc.ProductSubcategoryID
LEFT JOIN Production.ProductCategory c
    ON sc.ProductCategoryID = c.ProductCategoryID
LEFT JOIN inv
    ON p.ProductID = inv.sku_id
LEFT JOIN (SELECT sku_id, unit_cost FROM cost_latest WHERE rn = 1) cost
    ON p.ProductID = cost.sku_id
LEFT JOIN sales_12m
    ON p.ProductID = sales_12m.sku_id
;
"""

df = pd.read_sql(snapshot_sql, engine)

df.shape, df.columns.tolist()

((504, 8),
 ['sku_id',
  'sku_name',
  'category',
  'subcategory',
  'inventory_units',
  'unit_cost',
  'sales_units_12m',
  'revenue_12m'])

In [4]:
df = df.copy()

# Normalizar nulos
for col in ["category", "subcategory"]:
    df[col] = df[col].fillna("Unknown")

# Asegurar numéricos
num_cols = ["inventory_units", "unit_cost", "sales_units_12m", "revenue_12m"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# Capital invertido en inventario
df["inventory_capital"] = df["inventory_units"] * df["unit_cost"]

# COGS estimado 12m (proxy) = unidades vendidas * costo unitario
df["cogs_12m_est"] = df["sales_units_12m"] * df["unit_cost"]

# Gross profit 12m (proxy)
df["gross_profit_12m_est"] = df["revenue_12m"] - df["cogs_12m_est"]

# Precio unitario estimado
df["unit_price_est"] = np.where(
    df["sales_units_12m"] > 0,
    df["revenue_12m"] / df["sales_units_12m"],
    np.nan
)

# Días de inventario (basado en demanda 12m)
df["daily_units_12m"] = df["sales_units_12m"] / 365.0
df["days_inventory"] = np.where(
    df["daily_units_12m"] > 0,
    df["inventory_units"] / df["daily_units_12m"],
    np.inf
)

# Inventory ROIC (proxy) = gross profit / inventory capital
df["inventory_roic"] = np.where(
    df["inventory_capital"] > 0,
    df["gross_profit_12m_est"] / df["inventory_capital"],
    np.nan
)

# Flags
df["has_inventory_capital"] = df["inventory_capital"] > 0
df["has_sales_12m"] = df["sales_units_12m"] > 0

df[["inventory_capital","gross_profit_12m_est","inventory_roic","days_inventory"]].describe(include="all")

/Users/jorgeprax/sintelo-working-capital-mvp/.venv/lib/python3.9/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/jorgeprax/sintelo-working-capital-mvp/.venv/lib/python3.9/site-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


,inventory_capital,gross_profit_12m_est,inventory_roic,days_inventory
count,504.000000,504.000000,219.000000,504.000000
mean,38146.496094,11895.428110,1.863096,inf
std,82209.086370,57268.853143,7.206808,NaN
min,0.000000,-97709.427453,-11.813373,0.000000
25%,0.000000,0.000000,0.000000,145.301435
50%,0.000000,0.000000,0.051978,NaN
75%,37493.052475,775.226475,0.488622,NaN
max,616360.194000,532962.909420,75.645499,inf


In [5]:
analysis = df[df["inventory_capital"] > 0].copy()

# Filtrar ROIC razonable (evitar inf/nan)
analysis = analysis.replace([np.inf, -np.inf], np.nan)
analysis = analysis.dropna(subset=["inventory_roic"])

# Percentiles globales (sobre SKUs con capital)
p25, p50, p75 = np.nanpercentile(analysis["inventory_roic"], [25, 50, 75])

p25, p50, p75

(np.float64(0.0),
 np.float64(0.051977887535353914),
 np.float64(0.48862182364006623))

In [6]:
BENCH = float(p75)  # benchmark interno

analysis["benchmark_roic"] = BENCH

# Upside: si el SKU rindiera al benchmark interno (manteniendo capital constante)
analysis["upside_gross_profit"] = np.maximum(
    0.0,
    (analysis["benchmark_roic"] * analysis["inventory_capital"]) - analysis["gross_profit_12m_est"]
)

analysis["roic_gap"] = analysis["benchmark_roic"] - analysis["inventory_roic"]

analysis[["sku_id","sku_name","inventory_capital","inventory_roic","benchmark_roic","upside_gross_profit","days_inventory"]].head()

,sku_id,sku_name,inventory_capital,inventory_roic,benchmark_roic,upside_gross_profit,days_inventory
211,707,"Sport-100 Helmet, Red",3768.8544,16.233084,0.488622,0.000000,26.680203
212,708,"Sport-100 Helmet, Black",4239.9612,14.077065,0.488622,0.000000,29.337633
213,709,"Mountain Bike Socks, M",611.3340,0.000000,0.488622,298.711134,NaN
214,710,"Mountain Bike Socks, L",733.6008,0.000000,0.488622,358.453361,NaN
215,711,"Sport-100 Helmet, Blue",2826.6408,21.302845,0.488622,0.000000,19.437870


In [7]:
def weighted_roic(group: pd.DataFrame) -> float:
    cap = group["inventory_capital"].sum()
    if cap <= 0:
        return np.nan
    return group["gross_profit_12m_est"].sum() / cap

total_cap = analysis["inventory_capital"].sum()

by_cat = (
    analysis.groupby("category", as_index=False)
    .agg(
        skus=("sku_id","nunique"),
        inventory_capital=("inventory_capital","sum"),
        gross_profit_12m=("gross_profit_12m_est","sum"),
        upside=("upside_gross_profit","sum"),
        sales_units_12m=("sales_units_12m","sum"),
        revenue_12m=("revenue_12m","sum"),
    )
)
by_cat["inventory_roic"] = by_cat["gross_profit_12m"] / by_cat["inventory_capital"]
by_cat["capital_share"] = by_cat["inventory_capital"] / total_cap
by_cat["upside_pct_of_capital"] = by_cat["upside"] / by_cat["inventory_capital"]

by_cat = by_cat.sort_values("inventory_capital", ascending=False)

by_sub = (
    analysis.groupby(["category","subcategory"], as_index=False)
    .agg(
        skus=("sku_id","nunique"),
        inventory_capital=("inventory_capital","sum"),
        gross_profit_12m=("gross_profit_12m_est","sum"),
        upside=("upside_gross_profit","sum"),
        sales_units_12m=("sales_units_12m","sum"),
        revenue_12m=("revenue_12m","sum"),
    )
)
by_sub["inventory_roic"] = by_sub["gross_profit_12m"] / by_sub["inventory_capital"]
by_sub["capital_share"] = by_sub["inventory_capital"] / total_cap
by_sub["upside_pct_of_capital"] = by_sub["upside"] / by_sub["inventory_capital"]

by_sub = by_sub.sort_values("upside", ascending=False)

by_cat.head(10), by_sub.head(10)

(      category  skus  inventory_capital  gross_profit_12m        upside  \
 1        Bikes    97       1.462365e+07      4.904789e+06  5.012206e+06   
 3   Components    62       4.375837e+06      2.208036e+05  1.918187e+06   
 2     Clothing    32       1.381329e+05      1.741378e+05  1.255687e+05   
 0  Accessories    28       8.821176e+04      4.771426e+05  9.505665e+03   
 
    sales_units_12m   revenue_12m  inventory_roic  capital_share  \
 1            40934  4.159044e+07        0.335401       0.760625   
 3            13509  1.408792e+06        0.050460       0.227602   
 2            38725  1.156878e+06        1.260654       0.007185   
 0            47755  8.440939e+05        5.409059       0.004588   
 
    upside_pct_of_capital  
 1               0.342746  
 3               0.438359  
 2               0.909042  
 0               0.107760  ,
       category      subcategory  skus  inventory_capital  gross_profit_12m  \
 12       Bikes       Road Bikes    43       6.927376e+0

In [8]:
summary = {
    "snapshot_date": SNAPSHOT_DATE,
    "total_inventory_capital": float(total_cap),
    "gross_profit_12m_est": float(analysis["gross_profit_12m_est"].sum()),
    "current_inventory_roic": float(analysis["gross_profit_12m_est"].sum() / total_cap) if total_cap > 0 else None,
    "benchmark_inventory_roic_p75": float(BENCH),
    "potential_upside_gross_profit": float(analysis["upside_gross_profit"].sum()),
    "skus_in_scope": int(analysis["sku_id"].nunique()),
}

summary

{'snapshot_date': '2026-02-03',
 'total_inventory_capital': 19225834.0312,
 'gross_profit_12m_est': 5776873.398936002,
 'current_inventory_roic': 0.30047452763615856,
 'benchmark_inventory_roic_p75': 0.48862182364006623,
 'potential_upside_gross_profit': 7065467.215338863,
 'skus_in_scope': 219}

In [9]:
def to_py(obj):
    # Convierte numpy types a python types para JSON
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if obj is None:
        return None
    return obj

out_cat = DATA_OUT / f"ic_by_category_{SNAPSHOT_DATE}.csv"
out_sub = DATA_OUT / f"ic_by_subcategory_{SNAPSHOT_DATE}.csv"
out_sku = DATA_OUT / f"ic_by_sku_{SNAPSHOT_DATE}.csv"
out_ctx = DATA_OUT / f"context_{SNAPSHOT_DATE}.json"

by_cat.to_csv(out_cat, index=False)
by_sub.to_csv(out_sub, index=False)

# SKU table (para drill-down / decisiones)
sku_cols = [
    "sku_id","sku_name","category","subcategory",
    "inventory_units","unit_cost","inventory_capital",
    "sales_units_12m","revenue_12m","gross_profit_12m_est",
    "inventory_roic","benchmark_roic","roic_gap","upside_gross_profit",
    "days_inventory","has_sales_12m"
]
analysis[sku_cols].sort_values("upside_gross_profit", ascending=False).to_csv(out_sku, index=False)

context = {
    "summary": {k: to_py(v) for k, v in summary.items()},
    "benchmarks": {
        "p25": to_py(p25),
        "p50": to_py(p50),
        "p75": to_py(p75),
        "benchmark_used": to_py(BENCH),
    },
    "top_categories_by_capital": by_cat.head(10).to_dict(orient="records"),
    "top_subcategories_by_upside": by_sub.head(15).to_dict(orient="records"),
}

# Convert dict recursively
def convert_dict(d):
    if isinstance(d, dict):
        return {k: convert_dict(v) for k, v in d.items()}
    if isinstance(d, list):
        return [convert_dict(x) for x in d]
    return to_py(d)

out_ctx.write_text(json.dumps(convert_dict(context), indent=2), encoding="utf-8")

out_cat, out_sub, out_sku, out_ctx

(PosixPath('data/out/ic_by_category_2026-02-03.csv'),
 PosixPath('data/out/ic_by_subcategory_2026-02-03.csv'),
 PosixPath('data/out/ic_by_sku_2026-02-03.csv'),
 PosixPath('data/out/context_2026-02-03.json'))

In [10]:
summary["categories_in_scope"] = (
    df.loc[df["inventory_capital"] > 0, "category"]
      .nunique()
)

summary["subcategories_in_scope"] = (
    df.loc[df["inventory_capital"] > 0, "subcategory"]
      .nunique()
)

In [11]:
print([v for v in globals().keys() if v.startswith("by_")])

['by_cat', 'by_sub']


In [12]:
by_sku = (
    df
    .query("inventory_capital > 0")
    .assign(
        current_roic=lambda x: x["gross_profit_12m_est"] / x["inventory_capital"],
        upside_gp=lambda x: (summary["benchmark_inventory_roic_p75"] - x["current_roic"]).clip(lower=0) * x["inventory_capital"]
    )
    .sort_values("upside_gp", ascending=False)
)

In [13]:
# ============================
# Crear df_eval desde df base
# ============================

df_eval = df.copy()

# Solo SKUs con capital
df_eval = df_eval[df_eval["inventory_capital"] > 0].copy()

# Benchmark interno (P75)
p75_roic = df_eval["inventory_roic"].quantile(0.75)

# Upside económico por SKU
df_eval["upside_gross_profit"] = (
    (p75_roic - df_eval["inventory_roic"])
    * df_eval["inventory_capital"]
)

df_eval["upside_gross_profit"] = df_eval["upside_gross_profit"].clip(lower=0)

print("df_eval listo:", df_eval.shape)

df_eval listo: (219, 18)


In [14]:
# ============================
# Definir by_sub_prior correctamente
# ============================

# Base: solo SKUs con capital
tmp = df_eval[df_eval["inventory_capital"] > 0].copy()

# ROIC ponderado por capital
tmp["roic_x_cap"] = tmp["inventory_roic"] * tmp["inventory_capital"]

roic_w = (
    tmp.groupby(["category", "subcategory"], dropna=False)[["roic_x_cap", "inventory_capital"]]
    .sum()
    .reset_index()
)

roic_w["inventory_roic"] = roic_w["roic_x_cap"] / roic_w["inventory_capital"]

# Tabla principal
by_sub_prior = (
    tmp.groupby(["category", "subcategory"], dropna=False)
    .agg(
        inventory_capital=("inventory_capital", "sum"),
        upside_gross_profit=("upside_gross_profit", "sum"),
        skus=("inventory_capital", "size"),
    )
    .reset_index()
)

# Merge ROIC
by_sub_prior = by_sub_prior.merge(
    roic_w[["category", "subcategory", "inventory_roic"]],
    on=["category", "subcategory"],
    how="left"
)

# Upside relativo
by_sub_prior["upside_per_capital"] = (
    by_sub_prior["upside_gross_profit"] /
    by_sub_prior["inventory_capital"]
)

# Orden IC-style: por capital
by_sub_prior = by_sub_prior.sort_values(
    "inventory_capital",
    ascending=False
)


# ============================
# Debug prints (ya no falla)
# ============================

print("by_cat cols:", list(by_cat.columns))
print("by_sub cols:", list(by_sub.columns))
print("by_sku cols:", list(by_sku.columns))
print("by_sub_prior cols:", list(by_sub_prior.columns))

by_sub_prior.head(10)

by_cat cols: ['category', 'skus', 'inventory_capital', 'gross_profit_12m', 'upside', 'sales_units_12m', 'revenue_12m', 'inventory_roic', 'capital_share', 'upside_pct_of_capital']
by_sub cols: ['category', 'subcategory', 'skus', 'inventory_capital', 'gross_profit_12m', 'upside', 'sales_units_12m', 'revenue_12m', 'inventory_roic', 'capital_share', 'upside_pct_of_capital']
by_sku cols: ['sku_id', 'sku_name', 'category', 'subcategory', 'inventory_units', 'unit_cost', 'sales_units_12m', 'revenue_12m', 'inventory_capital', 'cogs_12m_est', 'gross_profit_12m_est', 'unit_price_est', 'daily_units_12m', 'days_inventory', 'inventory_roic', 'has_inventory_capital', 'has_sales_12m', 'current_roic', 'upside_gp']
by_sub_prior cols: ['category', 'subcategory', 'inventory_capital', 'upside_gross_profit', 'skus', 'inventory_roic', 'upside_per_capital']


,category,subcategory,inventory_capital,upside_gross_profit,skus,inventory_roic,upside_per_capital
12,Bikes,Road Bikes,6.927376e+06,2.400478e+06,43,0.170645,0.346521
11,Bikes,Mountain Bikes,4.771084e+06,1.497072e+06,32,0.702108,0.313780
13,Bikes,Touring Bikes,2.925193e+06,1.114656e+06,22,0.127463,0.381054
30,Components,Mountain Frames,1.671963e+06,7.611152e+05,6,0.033399,0.455222
34,Components,Wheels,1.059724e+06,5.173112e+05,14,0.000465,0.488156
25,Components,Cranksets,3.379897e+05,1.164136e+05,3,0.144193,0.344429
32,Components,Road Frames,2.873105e+05,1.418261e+05,1,-0.005012,0.493634
27,Components,Forks,2.173232e+05,1.061889e+05,3,0.000000,0.488622
28,Components,Handlebars,2.161643e+05,8.406701e+04,8,0.099719,0.388903
31,Components,Pedals,1.282443e+05,2.855963e+04,7,0.272643,0.222697


In [15]:
import pandas as pd
import numpy as np

def df_to_text(df, cols=None, n=10):
    x = df.copy()
    if cols:
        cols = [c for c in cols if c in x.columns]
        if cols:
            x = x[cols]
    return x.head(n).to_string(index=False)

def pick_first(existing_cols, candidates):
    for c in candidates:
        if c in existing_cols:
            return c
    return None

def pick_first_numeric(df):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    return num_cols[0] if num_cols else None

print("by_cat cols:", list(by_cat.columns))
print("by_sub cols:", list(by_sub.columns))

# --- detecta columnas (sin suposiciones) ---
cat_cols = set(by_cat.columns)
sub_cols = set(by_sub.columns)

cat_col_name = pick_first(cat_cols, ["category", "Category", "category_name"])
sub_col_name = pick_first(sub_cols, ["subcategory", "Subcategory", "subcategory_name"])

cat_cap_col = pick_first(cat_cols, ["capital", "inventory_capital", "inv_capital", "inventory_value"])
sub_cap_col = pick_first(sub_cols, ["capital", "inventory_capital", "inv_capital", "inventory_value"])

cat_up_col  = pick_first(cat_cols, ["upside", "upside_gross_profit", "potential_upside", "upside_profit"])
sub_up_col  = pick_first(sub_cols, ["upside", "upside_gross_profit", "potential_upside", "upside_profit"])

cat_roic_col = pick_first(cat_cols, ["roic", "current_roic", "inventory_roic"])
sub_roic_col = pick_first(sub_cols, ["roic", "current_roic", "inventory_roic"])

# --- decide sorting (prioridad: upside -> capital -> primera numérica) ---
cat_sort = cat_up_col or cat_cap_col or pick_first_numeric(by_cat)
sub_sort = sub_up_col or sub_cap_col or pick_first_numeric(by_sub)

if cat_sort is None:
    raise ValueError("by_cat no tiene columnas numéricas para ordenar.")
if sub_sort is None:
    raise ValueError("by_sub no tiene columnas numéricas para ordenar.")

# --- arma tablas “printables” ---
by_cat_sorted = by_cat.sort_values(by=cat_sort, ascending=False)
by_sub_sorted = by_sub.sort_values(by=sub_sort, ascending=False)

by_cat_cols_show = [c for c in [cat_col_name, cat_cap_col, cat_roic_col, cat_up_col] if c]
by_sub_cols_show = [c for c in [sub_col_name, sub_cap_col, sub_roic_col, sub_up_col] if c]

by_cat_printable = df_to_text(by_cat_sorted, cols=by_cat_cols_show, n=8)
by_sub_printable = df_to_text(by_sub_sorted, cols=by_sub_cols_show, n=12)

print("\n--- by_cat_printable ---")
print(by_cat_printable)
print("\n--- by_sub_printable ---")
print(by_sub_printable)

by_cat cols: ['category', 'skus', 'inventory_capital', 'gross_profit_12m', 'upside', 'sales_units_12m', 'revenue_12m', 'inventory_roic', 'capital_share', 'upside_pct_of_capital']
by_sub cols: ['category', 'subcategory', 'skus', 'inventory_capital', 'gross_profit_12m', 'upside', 'sales_units_12m', 'revenue_12m', 'inventory_roic', 'capital_share', 'upside_pct_of_capital']

--- by_cat_printable ---
   category  inventory_capital  inventory_roic       upside
      Bikes       1.462365e+07        0.335401 5.012206e+06
 Components       4.375837e+06        0.050460 1.918187e+06
   Clothing       1.381329e+05        1.260654 1.255687e+05
Accessories       8.821176e+04        5.409059 9.505665e+03

--- by_sub_printable ---
    subcategory  inventory_capital  inventory_roic       upside
     Road Bikes       6927375.9819        0.170645 2.400478e+06
 Mountain Bikes       4771083.8883        0.702108 1.497072e+06
  Touring Bikes       2925192.6622        0.127463 1.114656e+06
Mountain Frames    

In [16]:
import numpy as np

# ---------- helpers ----------
def pick_first(existing_cols, candidates):
    for c in candidates:
        if c in existing_cols:
            return c
    return None

def pick_first_numeric(df):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    return num_cols[0] if num_cols else None

def fmt_money(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "N/A"
    return f"{x:,.0f}"

def fmt_roic(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "N/A"
    return f"{x:.2f}x"

def fmt_pct(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "N/A"
    return f"{x:.1%}"

def format_table(df, key_col=None, cap_col=None, roic_col=None, up_col=None, sort_col=None, n=10):
    x = df.copy()

    # decide sort column
    if sort_col is None or sort_col not in x.columns:
        sort_col = up_col if up_col in x.columns else cap_col if cap_col in x.columns else pick_first_numeric(x)
    if sort_col is not None:
        x = x.sort_values(sort_col, ascending=False)

    # build output frame
    cols = [c for c in [key_col, cap_col, roic_col, up_col] if c and c in x.columns]
    if cols:
        x = x[cols].head(n)
    else:
        x = x.head(n)

    # apply formatting
    for c in x.columns:
        if cap_col and c == cap_col:
            x[c] = x[c].apply(fmt_money)
        if up_col and c == up_col:
            x[c] = x[c].apply(fmt_money)
        if roic_col and c == roic_col:
            x[c] = x[c].apply(fmt_roic)

    return x.to_string(index=False)

# ---------- detect columns ----------
cat_cols = set(by_cat.columns)
sub_cols = set(by_sub.columns)

cat_name = pick_first(cat_cols, ["category", "Category", "category_name"])
sub_name = pick_first(sub_cols, ["subcategory", "Subcategory", "subcategory_name"])

cat_cap  = pick_first(cat_cols, ["capital", "inventory_capital", "inv_capital", "inventory_value"])
sub_cap  = pick_first(sub_cols, ["capital", "inventory_capital", "inv_capital", "inventory_value"])

cat_roic = pick_first(cat_cols, ["roic", "current_roic", "inventory_roic"])
sub_roic = pick_first(sub_cols, ["roic", "current_roic", "inventory_roic"])

cat_up   = pick_first(cat_cols, ["upside", "upside_gross_profit", "potential_upside"])
sub_up   = pick_first(sub_cols, ["upside", "upside_gross_profit", "potential_upside"])

# ---------- summary fields (robust) ----------
skus_in_scope = summary.get("skus_in_scope", "N/A")
total_capital = summary.get("total_inventory_capital", np.nan)

gross_profit  = summary.get("gross_profit_12m_est", np.nan)
cur_roic      = summary.get("current_inventory_roic", np.nan)

p75_roic      = summary.get("benchmark_inventory_roic_p75", np.nan)
upside_gp     = summary.get("potential_upside_gross_profit", np.nan)

gross_profit_at_p75 = (gross_profit + upside_gp) if (np.isfinite(gross_profit) and np.isfinite(upside_gp)) else np.nan
gap_vs_p75 = (p75_roic - cur_roic) if (np.isfinite(p75_roic) and np.isfinite(cur_roic)) else np.nan
upside_pct = (upside_gp / total_capital) if (np.isfinite(upside_gp) and np.isfinite(total_capital) and total_capital != 0) else np.nan

# optional scope counts (si no existen, los calculamos)
categories_in_scope = summary.get("categories_in_scope", int(by_cat[cat_name].nunique()) if cat_name in by_cat.columns else "N/A")
subcategories_in_scope = summary.get("subcategories_in_scope", int(by_sub[sub_name].nunique()) if sub_name in by_sub.columns else "N/A")

# ---------- printable tables embedded ----------
by_cat_printable = format_table(
    by_cat,
    key_col=cat_name,
    cap_col=cat_cap,
    roic_col=cat_roic,
    up_col=cat_up,          # si no existe, no se muestra
    sort_col=cat_up or cat_cap,
    n=10
)

by_sub_printable = format_table(
    by_sub,
    key_col=sub_name,
    cap_col=sub_cap,
    roic_col=sub_roic,
    up_col=sub_up,          # si no existe, no se muestra
    sort_col=sub_up or sub_cap,
    n=15
)

# ---------- memo (printable) ----------
memo = f"""
INVENTORY ROIC EVALUATION MEMO
Snapshot: {SNAPSHOT_DATE}

1. SCOPE
- Inventory SKUs in scope (capital > 0): {skus_in_scope}
- Categories in scope: {categories_in_scope}
- Subcategories in scope: {subcategories_in_scope}
- Inventory capital deployed: {fmt_money(total_capital)}

2. CURRENT PERFORMANCE (Inventory Capital, proxy)
- Gross profit (12m, est.): {fmt_money(gross_profit)}
- Inventory ROIC (capital-weighted): {fmt_roic(cur_roic)}

3. INTERNAL BENCHMARK (SKU distribution)
- ROIC P75: {fmt_roic(p75_roic)}
- Gap vs P75: {fmt_roic(gap_vs_p75)}

4. UPSIDE POTENTIAL (Inventory Only, proxy)
- Gross profit at P75 (proxy): {fmt_money(gross_profit_at_p75)}
- Incremental upside vs current: {fmt_money(upside_gp)}
- Upside / inventory capital: {fmt_pct(upside_pct)}

5. CAPITAL & UPSIDE CONCENTRATION — CATEGORY
{by_cat_printable}

6. PRIORITY SUBCATEGORIES
{by_sub_printable}

NOTES
- Economic proxy for capital allocation decisions (not accounting ROIC).
- Benchmark uses internal percentiles (P75) to avoid external assumptions.
- Category/Subcategory tables are designed for weekly tracking and prioritization.
""".strip()

print(memo)

INVENTORY ROIC EVALUATION MEMO
Snapshot: 2026-02-03

1. SCOPE
- Inventory SKUs in scope (capital > 0): 219
- Categories in scope: 4
- Subcategories in scope: 35
- Inventory capital deployed: 19,225,834

2. CURRENT PERFORMANCE (Inventory Capital, proxy)
- Gross profit (12m, est.): 5,776,873
- Inventory ROIC (capital-weighted): 0.30x

3. INTERNAL BENCHMARK (SKU distribution)
- ROIC P75: 0.49x
- Gap vs P75: 0.19x

4. UPSIDE POTENTIAL (Inventory Only, proxy)
- Gross profit at P75 (proxy): 12,842,341
- Incremental upside vs current: 7,065,467
- Upside / inventory capital: 36.7%

5. CAPITAL & UPSIDE CONCENTRATION — CATEGORY
   category inventory_capital inventory_roic    upside
      Bikes        14,623,653          0.34x 5,012,206
 Components         4,375,837          0.05x 1,918,187
   Clothing           138,133          1.26x   125,569
Accessories            88,212          5.41x     9,506

6. PRIORITY SUBCATEGORIES
    subcategory inventory_capital inventory_roic    upside
     Road Bik

In [17]:
import numpy as np
import pandas as pd

def fmt_money(x):
    x = pd.to_numeric(x, errors="coerce")
    return "" if pd.isna(x) else f"{x:,.0f}"

def fmt_roic(x):
    x = pd.to_numeric(x, errors="coerce")
    return "" if pd.isna(x) else f"{x:.2f}x"

def fmt_pct(x):
    x = pd.to_numeric(x, errors="coerce")
    return "" if pd.isna(x) else f"{x:.1%}"

def df_to_text(df, cols, n=12, sort_by=None, ascending=False, formats=None):
    """
    formats: dict col-> ("money"|"roic"|"pct"|"raw")
    """
    d = df.copy()
    if sort_by and sort_by in d.columns:
        d = d.sort_values(sort_by, ascending=ascending)
    d = d.head(n)[cols].copy()

    formats = formats or {}
    for c in d.columns:
        f = formats.get(c, "raw")
        if f == "money":
            d[c] = d[c].apply(fmt_money)
        elif f == "roic":
            d[c] = d[c].apply(fmt_roic)
        elif f == "pct":
            d[c] = d[c].apply(fmt_pct)
        else:
            d[c] = d[c].astype(str)

    return d.to_string(index=False)

In [18]:
# ============================
# ROIC ponderado por categoría/subcategoría (sin apply)
# ============================

cap_col = "inventory_capital"
roic_col = "inventory_roic"
cat_col = "category"
sub_col = "subcategory"

# Validaciones rápidas
needed = [cap_col, roic_col, cat_col, sub_col]
missing = [c for c in needed if c not in df_eval.columns]
if missing:
    raise KeyError(f"Faltan columnas en df_eval: {missing}")

tmp = df_eval[df_eval[cap_col] > 0].copy()
tmp["roic_x_cap"] = tmp[roic_col] * tmp[cap_col]

# --- categoría ---
cat_w = (
    tmp.groupby(cat_col, dropna=False)
       .agg(roic_x_cap=("roic_x_cap", "sum"),
            cap_sum=(cap_col, "sum"))
       .reset_index()
)
cat_w["inventory_roic"] = cat_w["roic_x_cap"] / cat_w["cap_sum"]

by_cat = by_cat.merge(cat_w[[cat_col, "inventory_roic"]], on=cat_col, how="left")

# --- subcategoría ---
sub_w = (
    tmp.groupby([cat_col, sub_col], dropna=False)
       .agg(roic_x_cap=("roic_x_cap", "sum"),
            cap_sum=(cap_col, "sum"))
       .reset_index()
)
sub_w["inventory_roic"] = sub_w["roic_x_cap"] / sub_w["cap_sum"]

by_sub = by_sub.merge(sub_w[[cat_col, sub_col, "inventory_roic"]], on=[cat_col, sub_col], how="left")

In [19]:
# ---------------------------
# ENSURE REQUIRED COLUMNS
# ---------------------------

# 1) Asegura que by_cat tenga category + inventory_capital
if "inventory_capital" not in by_cat.columns:
    raise ValueError(f"by_cat no tiene 'inventory_capital'. Columnas: {list(by_cat.columns)}")

# 2) Asegura upside_gross_profit (si tu columna se llama distinto, mapea)
if "upside_gross_profit" not in by_cat.columns:
    # intenta alternativas comunes
    for alt in ["upside", "potential_upside_gross_profit", "upside_gp", "upside_profit"]:
        if alt in by_cat.columns:
            by_cat = by_cat.rename(columns={alt: "upside_gross_profit"})
            break
if "upside_gross_profit" not in by_cat.columns:
    # si no existe, crea 0 para poder imprimir
    by_cat["upside_gross_profit"] = 0.0

# 3) Asegura inventory_roic
if "inventory_roic" not in by_cat.columns:
    # Si tienes gross_profit_est, se puede inferir ROIC = GP / capital
    if "gross_profit_est" in by_cat.columns:
        by_cat["inventory_roic"] = by_cat["gross_profit_est"] / by_cat["inventory_capital"].replace(0, np.nan)
    # Si tienes roic_x_cap y cap_sum, también
    elif "roic_x_cap" in by_cat.columns and "cap_sum" in by_cat.columns:
        by_cat["inventory_roic"] = by_cat["roic_x_cap"] / by_cat["cap_sum"].replace(0, np.nan)
    else:
        # último recurso: calcula desde df_eval (garantizado si existe)
        tmp = df_eval[df_eval[cap_col] > 0].copy()
        tmp["roic_x_cap"] = tmp[roic_col] * tmp[cap_col]
        cat_w = tmp.groupby(cat_col, dropna=False).agg(
            roic_x_cap=("roic_x_cap", "sum"),
            cap_sum=(cap_col, "sum")
        ).reset_index()
        cat_w["inventory_roic"] = cat_w["roic_x_cap"] / cat_w["cap_sum"].replace(0, np.nan)

        by_cat = by_cat.merge(cat_w[[cat_col, "inventory_roic"]], on=cat_col, how="left")

# 4) Columnas derivadas para el memo
total_cap = by_cat["inventory_capital"].sum()
by_cat["capital_share"] = np.where(total_cap > 0, by_cat["inventory_capital"] / total_cap, 0.0)
by_cat["upside_per_capital"] = by_cat["upside_gross_profit"] / by_cat["inventory_capital"].replace(0, np.nan)

In [20]:
# ---------------------------
# ENSURE REQUIRED COLUMNS (SUB)
# ---------------------------

if "inventory_capital" not in by_sub.columns:
    raise ValueError(f"by_sub no tiene 'inventory_capital'. Columnas: {list(by_sub.columns)}")

if "upside_gross_profit" not in by_sub.columns:
    for alt in ["upside", "potential_upside_gross_profit", "upside_gp", "upside_profit"]:
        if alt in by_sub.columns:
            by_sub = by_sub.rename(columns={alt: "upside_gross_profit"})
            break
if "upside_gross_profit" not in by_sub.columns:
    by_sub["upside_gross_profit"] = 0.0

if "inventory_roic" not in by_sub.columns:
    tmp = df_eval[df_eval[cap_col] > 0].copy()
    tmp["roic_x_cap"] = tmp[roic_col] * tmp[cap_col]
    sub_w = tmp.groupby([cat_col, sub_col], dropna=False).agg(
        roic_x_cap=("roic_x_cap", "sum"),
        cap_sum=(cap_col, "sum")
    ).reset_index()
    sub_w["inventory_roic"] = sub_w["roic_x_cap"] / sub_w["cap_sum"].replace(0, np.nan)

    by_sub = by_sub.merge(sub_w[[cat_col, sub_col, "inventory_roic"]], on=[cat_col, sub_col], how="left")

by_sub["upside_per_capital"] = by_sub["upside_gross_profit"] / by_sub["inventory_capital"].replace(0, np.nan)

In [21]:
from pathlib import Path
import pandas as pd
import numpy as np

# ---------- Output folder ----------
OUT_DIR = Path("reports")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Format helpers ----------
def fmt_money(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return ""
    return f"{float(x):,.0f}"

def fmt_roic(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return ""
    return f"{float(x):.2f}x"

def fmt_pct(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return ""
    return f"{float(x):.1%}"

def df_to_text(df, cols, n=12, sort_by=None, ascending=False, formats=None):
    d = df.copy()
    if sort_by and sort_by in d.columns:
        d = d.sort_values(sort_by, ascending=ascending)
    d = d.head(n)[cols].copy()

    formats = formats or {}
    for c in d.columns:
        f = formats.get(c, None)
        if f == "money":
            d[c] = d[c].apply(fmt_money)
        elif f == "roic":
            d[c] = d[c].apply(fmt_roic)
        elif f == "pct":
            d[c] = d[c].apply(fmt_pct)
        else:
            d[c] = d[c].fillna("").astype(str)
    return d.to_string(index=False)

# ---------- Robust pulls from summary ----------
skus_in_scope = summary["skus_in_scope"]
categories_in_scope = summary.get("categories_in_scope", by_cat["category"].nunique() if "category" in by_cat.columns else "N/A")
subcategories_in_scope = summary.get("subcategories_in_scope", by_sub["subcategory"].nunique() if "subcategory" in by_sub.columns else "N/A")

total_cap = summary["total_inventory_capital"]
gross_profit = summary["gross_profit_12m_est"]
current_roic = summary["current_inventory_roic"]
p75_roic = summary["benchmark_inventory_roic_p75"]
gap = p75_roic - current_roic
upside = summary["potential_upside_gross_profit"]
gp_at_p75 = gross_profit + upside

# ---------- Build printable tables ----------
# Expect columns: category, inventory_capital, inventory_roic, upside_gross_profit, upside_per_capital, capital_share
# (your "ensure columns" block should have created these)
cat_table = df_to_text(
    by_cat,
    cols=["category","inventory_capital","inventory_roic","upside_gross_profit","upside_per_capital","capital_share"],
    n=10,
    sort_by="inventory_capital",
    ascending=False,
    formats={
        "category": "raw",
        "inventory_capital": "money",
        "inventory_roic": "roic",
        "upside_gross_profit": "money",
        "upside_per_capital": "pct",
        "capital_share": "pct",
    }
)

sub_table = df_to_text(
    by_sub,
    cols=["category","subcategory","inventory_capital","inventory_roic","upside_gross_profit","upside_per_capital"],
    n=15,
    sort_by="inventory_capital",
    ascending=False,
    formats={
        "category": "raw",
        "subcategory": "raw",
        "inventory_capital": "money",
        "inventory_roic": "roic",
        "upside_gross_profit": "money",
        "upside_per_capital": "pct",
    }
)

# ---------- Memo text ----------
memo = f"""
INVENTORY ROIC EVALUATION MEMO
Snapshot: {SNAPSHOT_DATE}

1. SCOPE
- Inventory SKUs in scope (capital > 0): {skus_in_scope}
- Categories in scope: {categories_in_scope}
- Subcategories in scope: {subcategories_in_scope}
- Inventory capital deployed: {fmt_money(total_cap)}

2. CURRENT PERFORMANCE (Inventory Capital, proxy)
- Gross profit (12m, est.): {fmt_money(gross_profit)}
- Inventory ROIC (capital-weighted): {fmt_roic(current_roic)}

3. INTERNAL BENCHMARK (SKU distribution)
- ROIC P75: {fmt_roic(p75_roic)}
- Gap vs P75: {fmt_roic(gap)}

4. UPSIDE POTENTIAL (Inventory Only, proxy)
- Gross profit at P75 (proxy): {fmt_money(gp_at_p75)}
- Incremental upside vs current: {fmt_money(upside)}
- Upside / inventory capital: {fmt_pct(upside / total_cap if total_cap else np.nan)}

5. CAPITAL & UPSIDE CONCENTRATION — CATEGORY
{cat_table}

6. PRIORITY SUBCATEGORIES (capital-ordered)
{sub_table}

NOTES
- Economic proxy for capital allocation decisions (not accounting ROIC).
- Benchmark uses internal percentiles (P75) to avoid external assumptions.
""".strip()

print(memo)

# ---------- Save to file (print-ready) ----------
memo_path = OUT_DIR / f"inventory_roic_memo_{SNAPSHOT_DATE}.txt"
memo_path.write_text(memo + "\n", encoding="utf-8")
print(f"\nSaved memo -> {memo_path}")

INVENTORY ROIC EVALUATION MEMO
Snapshot: 2026-02-03

1. SCOPE
- Inventory SKUs in scope (capital > 0): 219
- Categories in scope: 4
- Subcategories in scope: 35
- Inventory capital deployed: 19,225,834

2. CURRENT PERFORMANCE (Inventory Capital, proxy)
- Gross profit (12m, est.): 5,776,873
- Inventory ROIC (capital-weighted): 0.30x

3. INTERNAL BENCHMARK (SKU distribution)
- ROIC P75: 0.49x
- Gap vs P75: 0.19x

4. UPSIDE POTENTIAL (Inventory Only, proxy)
- Gross profit at P75 (proxy): 12,842,341
- Incremental upside vs current: 7,065,467
- Upside / inventory capital: 36.7%

5. CAPITAL & UPSIDE CONCENTRATION — CATEGORY
   category inventory_capital inventory_roic upside_gross_profit upside_per_capital capital_share
      Bikes        14,623,653          0.34x           5,012,206              34.3%         76.1%
 Components         4,375,837          0.05x           1,918,187              43.8%         22.8%
   Clothing           138,133          1.26x             125,569              90

In [35]:
import json
from pathlib import Path
from datetime import date
import pandas as pd
import numpy as np

# =========================
# 0) INPUT
# =========================
assert "df_eval" in globals() or "df" in globals(), "No existe df_eval ni df. Corre las celdas anteriores primero."
df_eval = (df_eval if "df_eval" in globals() else df).copy()

# Normaliza nulos (evita que truene groupby / prints)
for c in ["category", "subcategory"]:
    if c in df_eval.columns:
        df_eval[c] = df_eval[c].fillna("Unknown")

# Asegura numéricos clave
for c in ["inventory_capital", "gross_profit_12m_est", "inventory_roic", "upside_gross_profit"]:
    if c in df_eval.columns:
        df_eval[c] = pd.to_numeric(df_eval[c], errors="coerce")

# Filtra alcance
df_scope = df_eval[df_eval["inventory_capital"].fillna(0) > 0].copy()

total_capital = float(df_scope["inventory_capital"].sum())
if not np.isfinite(total_capital) or total_capital <= 0:
    raise RuntimeError("inventory_capital_total <= 0. Revisa datos/filtros.")

# =========================
# 1) FACTS (core)
# =========================
facts = {}
facts["company_name"] = "AdventureWorks Demo"
facts["prepared_by"] = "Sintelo"
facts["snapshot_date"] = str(date.today())

facts["inventory_skus_in_scope"] = int(df_scope.shape[0])
facts["categories_in_scope"] = int(df_scope["category"].nunique()) if "category" in df_scope.columns else 0
facts["subcategories_in_scope"] = int(df_scope["subcategory"].nunique()) if "subcategory" in df_scope.columns else 0

facts["inventory_capital_total"] = float(total_capital)

gp_total = float(np.nan_to_num(df_scope["gross_profit_12m_est"].sum(), nan=0.0))
facts["gross_profit_12m_est_total"] = float(gp_total)

facts["inventory_roic_weighted"] = float(gp_total / total_capital)

# Benchmark P75 (SKU distribution)
p75 = float(df_scope["inventory_roic"].replace([np.inf, -np.inf], np.nan).dropna().quantile(0.75))
facts["benchmark_roic_p75"] = p75
facts["gap_vs_p75"] = float(p75 - facts["inventory_roic_weighted"])

# Upside (proxy)
facts["gross_profit_at_p75"] = float(total_capital * p75)
facts["upside_vs_current"] = float(facts["gross_profit_at_p75"] - gp_total)
facts["upside_over_capital_pct"] = float(facts["upside_vs_current"] / total_capital)

# =========================
# 2) CONCENTRACIÓN (Category/Subcategory)
# =========================
# Category table
if "category" in df_scope.columns:
    by_cat = (
        df_scope.groupby("category", dropna=False)
        .agg(
            inventory_capital=("inventory_capital", "sum"),
            gross_profit_12m_est=("gross_profit_12m_est", "sum"),
            upside_gross_profit=("upside_gross_profit", "sum") if "upside_gross_profit" in df_scope.columns else ("inventory_capital", "size")
        )
        .reset_index()
    )
    by_cat["inventory_roic"] = by_cat["gross_profit_12m_est"] / by_cat["inventory_capital"]
    by_cat["capital_share"] = by_cat["inventory_capital"] / total_capital
    if "upside_gross_profit" in by_cat.columns:
        by_cat["upside_per_capital"] = by_cat["upside_gross_profit"] / by_cat["inventory_capital"]
    facts["top_categories_by_capital"] = (
        by_cat.sort_values("inventory_capital", ascending=False).head(10).to_dict(orient="records")
    )

# Subcategory by capital
if "category" in df_scope.columns and "subcategory" in df_scope.columns:
    by_sub_cap = (
        df_scope.groupby(["category","subcategory"], dropna=False)
        .agg(
            inventory_capital=("inventory_capital", "sum"),
            gross_profit_12m_est=("gross_profit_12m_est", "sum"),
            upside_gross_profit=("upside_gross_profit", "sum") if "upside_gross_profit" in df_scope.columns else ("inventory_capital", "size")
        )
        .reset_index()
    )
    by_sub_cap["inventory_roic"] = by_sub_cap["gross_profit_12m_est"] / by_sub_cap["inventory_capital"]
    if "upside_gross_profit" in by_sub_cap.columns:
        by_sub_cap["upside_per_capital"] = by_sub_cap["upside_gross_profit"] / by_sub_cap["inventory_capital"]

    facts["top_subcategories_by_capital"] = (
        by_sub_cap.sort_values("inventory_capital", ascending=False).head(15).to_dict(orient="records")
    )
    # Subcategory by upside (si existe)
    if "upside_gross_profit" in by_sub_cap.columns:
        facts["top_subcategories_by_upside"] = (
            by_sub_cap.sort_values("upside_gross_profit", ascending=False).head(15).to_dict(orient="records")
        )

# =========================
# 3) WRITE (a prueba de cwd)
# =========================
out_dir = Path.cwd() / "reports"
out_dir.mkdir(exist_ok=True)

out_path = out_dir / f"facts_pack_{facts['snapshot_date']}.json"
out_path.write_text(json.dumps(facts, indent=2, ensure_ascii=False), encoding="utf-8")

print("✅ FACTS PACK generado:", out_path)
print("Archivos en reports:", [p.name for p in out_dir.glob("*facts_pack*.json")])

✅ FACTS PACK generado: /Users/jorgeprax/sintelo-working-capital-mvp/notebooks/reports/facts_pack_2026-02-18.json
Archivos en reports: ['facts_pack_2026-02-18.json']


In [43]:
# ============================================
# INVENTORY CAPITAL MEMO (IC-grade, executable)
# Python 3.9 compatible
# Inputs: facts_pack JSON (only source of truth)
# Output: printable .txt memo in /notebooks/reports
# ============================================

import json
from pathlib import Path
from typing import Optional, List, Dict, Any
from datetime import datetime

import pandas as pd


# ----------------------------
# Format helpers (Word-safe)
# ----------------------------
def money(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "No disponible"
    try:
        return f"{float(x):,.0f}"
    except Exception:
        return "No disponible"

def pct(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "No disponible"
    try:
        return f"{float(x):.1%}"
    except Exception:
        return "No disponible"

def roic_x(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "No disponible"
    try:
        return f"{float(x):.2f}x"
    except Exception:
        return "No disponible"

def safe_str(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "No disponible"
    s = str(x)
    return s if s.strip() else "No disponible"

def df_to_text(df: pd.DataFrame, cols: List[str], n: int, sort_by: str, ascending: bool, formats: Dict[str, str]) -> str:
    if df is None or len(df) == 0:
        return "No disponible"

    d = df.copy()
    if sort_by in d.columns:
        d = d.sort_values(sort_by, ascending=ascending)

    cols_exist = [c for c in cols if c in d.columns]
    if not cols_exist:
        return "No disponible"

    d = d.head(n)[cols_exist].copy()

    for c in cols_exist:
        fmt = formats.get(c, "raw")
        if fmt == "money":
            d[c] = d[c].apply(lambda v: money(v))
        elif fmt == "pct":
            d[c] = d[c].apply(lambda v: pct(v))
        elif fmt == "roic":
            d[c] = d[c].apply(lambda v: roic_x(v))
        else:
            d[c] = d[c].apply(lambda v: safe_str(v))

    return d.to_string(index=False)


# ----------------------------
# IC rules (from your prompt)
# ----------------------------
def reduction_pct_from_gap_ratio(gap_ratio: float) -> float:
    # gap_ratio = (benchmark - roic) / benchmark
    if gap_ratio is None or pd.isna(gap_ratio):
        return 0.0
    if gap_ratio >= 0.75:
        return 0.60
    if 0.50 <= gap_ratio < 0.75:
        return 0.45
    if 0.25 <= gap_ratio < 0.50:
        return 0.30
    if 0.10 <= gap_ratio < 0.25:
        return 0.15
    return 0.0


def generate_inventory_capital_memo(
    facts_path: str = "reports/facts_pack_2026-02-18.json",
    out_dir: str = "reports",
    out_filename: Optional[str] = None,
    top_n_subcats_for_plan: int = 12,
):
    facts_path = Path(facts_path)
    if not facts_path.exists():
        raise FileNotFoundError(f"No existe facts pack: {facts_path.resolve()}")

    facts: Dict[str, Any] = json.loads(facts_path.read_text(encoding="utf-8"))

    # ----------------------------
    # Base facts (only from JSON)
    # ----------------------------
    company_name = safe_str(facts.get("company_name"))
    prepared_by = safe_str(facts.get("prepared_by", "Sintelo"))
    snapshot_date = safe_str(facts.get("snapshot_date"))

    skus_in_scope = facts.get("inventory_skus_in_scope")
    categories_in_scope = facts.get("categories_in_scope")
    subcategories_in_scope = facts.get("subcategories_in_scope")
    total_capital = facts.get("inventory_capital_total")

    gp_current = facts.get("gross_profit_12m_est_total")
    roic_weighted = facts.get("inventory_roic_weighted")

    p75 = facts.get("benchmark_roic_p75")
    gap_vs_p75 = facts.get("gap_vs_p75")

    gp_at_p75 = facts.get("gross_profit_at_p75")
    upside = facts.get("upside_vs_current")
    upside_pct = facts.get("upside_over_capital_pct")

    # ----------------------------
    # Tables from JSON (lists)
    # ----------------------------
    cat_rows = facts.get("top_categories_by_capital", []) or []
    sub_cap_rows = facts.get("top_subcategories_by_capital", []) or []
    sub_up_rows = facts.get("top_subcategories_by_upside", []) or []

    df_cat = pd.DataFrame(cat_rows)
    df_sub_cap = pd.DataFrame(sub_cap_rows)
    df_sub_up = pd.DataFrame(sub_up_rows)

    # Normalize expected column names for printing
    # (We don't create new numbers; we just rename if they exist)
    ren_cat = {
        "upside_gross_profit": "upside_gp",
        "upside_per_capital": "upside_per_cap",
        "capital_share": "cap_share",
    }
    ren_sub = {
        "upside_gross_profit": "upside_gp",
        "upside_per_capital": "upside_per_cap",
    }
    if len(df_cat) > 0:
        df_cat = df_cat.rename(columns=ren_cat)
    if len(df_sub_cap) > 0:
        df_sub_cap = df_sub_cap.rename(columns=ren_sub)
    if len(df_sub_up) > 0:
        df_sub_up = df_sub_up.rename(columns=ren_sub)

    # ----------------------------
    # Build Capital Reallocation Plan (from subcategory tables only)
    # ----------------------------
    benchmark = float(p75) if p75 is not None and not pd.isna(p75) else None
    if benchmark is None or benchmark <= 0:
        # Can't do IC plan without benchmark
        plan_note = "No disponible"
        df_reduce = pd.DataFrame()
        df_increase = pd.DataFrame()
        capital_released = 0.0
        capital_reassigned = 0.0
        residual_capital = 0.0
        inc_profit_total = 0.0
        new_roic = None
    else:
        # Use a pool of subcategories: combine cap-ordered and upside-ordered (dedupe)
        pool = pd.concat([df_sub_cap, df_sub_up], ignore_index=True) if (len(df_sub_cap) or len(df_sub_up)) else pd.DataFrame()
        if len(pool) == 0:
            plan_note = "No disponible"
            df_reduce = pd.DataFrame()
            df_increase = pd.DataFrame()
            capital_released = 0.0
            capital_reassigned = 0.0
            residual_capital = 0.0
            inc_profit_total = 0.0
            new_roic = None
        else:
            # Ensure minimal columns exist
            for col in ["category", "subcategory", "inventory_capital", "inventory_roic"]:
                if col not in pool.columns:
                    pool[col] = None

            pool = pool.drop_duplicates(subset=["category", "subcategory"]).copy()

            # Coerce numeric
            pool["inventory_capital"] = pd.to_numeric(pool["inventory_capital"], errors="coerce")
            pool["inventory_roic"] = pd.to_numeric(pool["inventory_roic"], errors="coerce")

            pool = pool.dropna(subset=["inventory_capital", "inventory_roic"])
            pool = pool[pool["inventory_capital"] > 0].copy()

            # Limit size for plan (still deterministic and based on facts pack)
            # Priority: high capital first (capital drives materiality)
            pool = pool.sort_values("inventory_capital", ascending=False).head(top_n_subcats_for_plan).copy()

            # Classify and compute gap_ratio
            pool["gap_ratio"] = (benchmark - pool["inventory_roic"]) / benchmark

            # Reduce set: roic < benchmark
            reduce_set = pool[pool["inventory_roic"] < benchmark].copy()
            reduce_set["reduce_pct"] = reduce_set["gap_ratio"].apply(reduction_pct_from_gap_ratio)
            reduce_set["capital_to_reduce"] = reduce_set["inventory_capital"] * reduce_set["reduce_pct"]

            # Increase set: roic > benchmark (rank by absolute ROIC desc, per prompt)
            inc_set = pool[pool["inventory_roic"] > benchmark].copy()
            inc_set = inc_set.sort_values("inventory_roic", ascending=False).copy()

            # Total released capital
            capital_released = float(reduce_set["capital_to_reduce"].sum()) if len(reduce_set) else 0.0

            # Constraints for increases:
            # - max increase per subcategory: 100% of its current capital
            # - subcategory cannot exceed 50% of total portfolio capital
            total_cap = float(total_capital) if total_capital is not None else 0.0
            max_share_cap = 0.50 * total_cap if total_cap > 0 else float("inf")

            inc_set["max_increase_100pct"] = inc_set["inventory_capital"] * 1.0
            inc_set["max_after_50pct_rule"] = (max_share_cap - inc_set["inventory_capital"]).clip(lower=0)
            inc_set["max_increase_allowed"] = inc_set[["max_increase_100pct", "max_after_50pct_rule"]].min(axis=1)

            # Allocate released capital in ROIC-desc order, respecting constraints
            remaining = capital_released
            alloc = []
            for _, r in inc_set.iterrows():
                if remaining <= 0:
                    alloc.append(0.0)
                    continue
                cap_allow = float(r["max_increase_allowed"]) if not pd.isna(r["max_increase_allowed"]) else 0.0
                add = min(remaining, max(0.0, cap_allow))
                alloc.append(add)
                remaining -= add

            inc_set["capital_to_increase"] = alloc
            capital_reassigned = float(inc_set["capital_to_increase"].sum()) if len(inc_set) else 0.0
            residual_capital = float(capital_released - capital_reassigned)

            # Source ROIC (weighted) across reduction origins for incremental profit calc
            if len(reduce_set) and float(reduce_set["capital_to_reduce"].sum()) > 0:
                source_roic = float(
                    (reduce_set["inventory_roic"] * reduce_set["capital_to_reduce"]).sum()
                    / reduce_set["capital_to_reduce"].sum()
                )
            else:
                source_roic = float(roic_weighted) if roic_weighted is not None else 0.0

            # Incremental profit: sum(cap_to_increase * (dest_roic - source_roic))
            if len(inc_set):
                inc_profit_total = float((inc_set["capital_to_increase"] * (inc_set["inventory_roic"] - source_roic)).sum())
            else:
                inc_profit_total = 0.0

            # New portfolio ROIC:
            # new_gross_profit = current_gp + incremental_profit
            # total_capital unchanged if residual is "reduced net inventory"; if residual treated as withdrawn, capital reduces.
            # We'll treat residual as "withdraw if no destination superior" (preferred). That means capital_after = total_cap - residual.
            current_gp = float(gp_current) if gp_current is not None else 0.0
            total_cap_after = total_cap - residual_capital if total_cap > 0 else total_cap
            new_gp = current_gp + inc_profit_total
            new_roic = (new_gp / total_cap_after) if total_cap_after and total_cap_after > 0 else None

            # Prepare tables for memo
            df_reduce = reduce_set[[
                "category", "subcategory", "inventory_capital", "inventory_roic", "gap_ratio", "reduce_pct", "capital_to_reduce"
            ]].copy()

            df_increase = inc_set[[
                "category", "subcategory", "inventory_capital", "inventory_roic", "capital_to_increase", "max_increase_allowed"
            ]].copy()

            plan_note = "Deterministico. Basado exclusivamente en facts pack y constraints institucionales."

    # ----------------------------
    # 6.5 New Portfolio Allocation table
    # ----------------------------
    allocation_rows = []
    if benchmark is not None and isinstance(benchmark, float) and len(df_sub_cap) > 0:
        # Build from pool (same subset used for plan, if available)
        base = pd.concat([df_sub_cap, df_sub_up], ignore_index=True) if (len(df_sub_cap) or len(df_sub_up)) else pd.DataFrame()
        if len(base) > 0:
            base = base.drop_duplicates(subset=["category","subcategory"]).copy()
            base["inventory_capital"] = pd.to_numeric(base.get("inventory_capital"), errors="coerce")
            base["inventory_roic"] = pd.to_numeric(base.get("inventory_roic"), errors="coerce")
            base = base.dropna(subset=["inventory_capital","inventory_roic"])
            base = base[base["inventory_capital"] > 0].copy()

            # Create maps of reductions/increases
            red_map = {}
            inc_map = {}
            if "df_reduce" in locals() and isinstance(df_reduce, pd.DataFrame) and len(df_reduce) > 0:
                for _, r in df_reduce.iterrows():
                    red_map[(r["category"], r["subcategory"])] = float(r["capital_to_reduce"])
            if "df_increase" in locals() and isinstance(df_increase, pd.DataFrame) and len(df_increase) > 0:
                for _, r in df_increase.iterrows():
                    inc_map[(r["category"], r["subcategory"])] = float(r["capital_to_increase"])

            for _, r in base.iterrows():
                k = (r["category"], r["subcategory"])
                before = float(r["inventory_capital"])
                roicv = float(r["inventory_roic"])
                reduced = red_map.get(k, 0.0)
                increased = inc_map.get(k, 0.0)
                after = before - reduced + increased
                allocation_rows.append({
                    "subcategory": f"{r['category']} / {r['subcategory']}",
                    "capital_before": before,
                    "capital_after": after,
                    "roic": roicv,
                })

    df_alloc = pd.DataFrame(allocation_rows)
    if len(df_alloc) > 0:
        df_alloc = df_alloc.sort_values("capital_before", ascending=False).copy()

    # ----------------------------
    # Printable blocks
    # ----------------------------
    cat_table = df_to_text(
        df_cat,
        cols=["category","inventory_capital","inventory_roic","upside_gp","upside_per_cap","cap_share"],
        n=10,
        sort_by="inventory_capital",
        ascending=False,
        formats={
            "category":"raw",
            "inventory_capital":"money",
            "inventory_roic":"roic",
            "upside_gp":"money",
            "upside_per_cap":"pct",
            "cap_share":"pct",
        }
    )

    sub_cap_table = df_to_text(
        df_sub_cap,
        cols=["category","subcategory","inventory_capital","inventory_roic","upside_gp","upside_per_cap"],
        n=15,
        sort_by="inventory_capital",
        ascending=False,
        formats={
            "category":"raw",
            "subcategory":"raw",
            "inventory_capital":"money",
            "inventory_roic":"roic",
            "upside_gp":"money",
            "upside_per_cap":"pct",
        }
    )

    sub_up_table = df_to_text(
        df_sub_up,
        cols=["category","subcategory","inventory_capital","inventory_roic","upside_gp","upside_per_cap"],
        n=15,
        sort_by="upside_gp" if ("upside_gp" in df_sub_up.columns) else "inventory_capital",
        ascending=False,
        formats={
            "category":"raw",
            "subcategory":"raw",
            "inventory_capital":"money",
            "inventory_roic":"roic",
            "upside_gp":"money",
            "upside_per_cap":"pct",
        }
    )

    reduce_table = df_to_text(
        df_reduce if "df_reduce" in locals() else pd.DataFrame(),
        cols=["category","subcategory","inventory_capital","inventory_roic","gap_ratio","reduce_pct","capital_to_reduce"],
        n=20,
        sort_by="capital_to_reduce",
        ascending=False,
        formats={
            "category":"raw",
            "subcategory":"raw",
            "inventory_capital":"money",
            "inventory_roic":"roic",
            "gap_ratio":"pct",
            "reduce_pct":"pct",
            "capital_to_reduce":"money",
        }
    )

    increase_table = df_to_text(
        df_increase if "df_increase" in locals() else pd.DataFrame(),
        cols=["category","subcategory","inventory_capital","inventory_roic","capital_to_increase","max_increase_allowed"],
        n=20,
        sort_by="capital_to_increase",
        ascending=False,
        formats={
            "category":"raw",
            "subcategory":"raw",
            "inventory_capital":"money",
            "inventory_roic":"roic",
            "capital_to_increase":"money",
            "max_increase_allowed":"money",
        }
    )

    alloc_table = df_to_text(
        df_alloc,
        cols=["subcategory","capital_before","capital_after","roic"],
        n=25,
        sort_by="capital_before",
        ascending=False,
        formats={
            "subcategory":"raw",
            "capital_before":"money",
            "capital_after":"money",
            "roic":"roic",
        }
    )

    # ----------------------------
    # Priority Actions (3–5) from top increases + top reductions
    # ----------------------------
    priority_actions = []
    if "df_reduce" in locals() and isinstance(df_reduce, pd.DataFrame) and len(df_reduce) > 0:
        rtop = df_reduce.sort_values("capital_to_reduce", ascending=False).head(3)
        for _, r in rtop.iterrows():
            priority_actions.append({
                "area": f"{r['category']} / {r['subcategory']}",
                "capital": r["inventory_capital"],
                "roic": r["inventory_roic"],
                "action": "Reducir exposicion. Liberar capital improductivo mediante liquidacion selectiva y reduccion de futuras compras en SKUs con ROIC inferior al benchmark.",
                "impact_capital": r["capital_to_reduce"],
                "impact_upside": None,  # not inventing per-subcat upside unless present
                "risk": "Validar cobertura de SKUs core antes de reduccion.",
            })

    if "df_increase" in locals() and isinstance(df_increase, pd.DataFrame) and len(df_increase) > 0:
        itop = df_increase.sort_values("capital_to_increase", ascending=False).head(2)
        for _, r in itop.iterrows():
            priority_actions.append({
                "area": f"{r['category']} / {r['subcategory']}",
                "capital": r["inventory_capital"],
                "roic": r["inventory_roic"],
                "action": "Aumentar exposicion. Reasignar capital liberado hacia esta subcategoria por ROIC superior al benchmark, respetando constraints de asignacion.",
                "impact_capital": r["capital_to_increase"],
                "impact_upside": None,
                "risk": "Asegurar capacidad operativa y abastecimiento antes de aumento.",
            })

    priority_actions = priority_actions[:5]

    actions_txt_lines = []
    if not priority_actions:
        actions_txt_lines.append("No disponible")
    else:
        for i, a in enumerate(priority_actions, 1):
            actions_txt_lines.append(f"{i}. Area: {a['area']}")
            actions_txt_lines.append(f"   Capital en alcance: {money(a['capital'])}")
            actions_txt_lines.append(f"   ROIC: {roic_x(a['roic'])} vs benchmark {roic_x(p75)}")
            actions_txt_lines.append(f"   Accion recomendada: {a['action']}")
            actions_txt_lines.append(f"   Impacto esperado:")
            actions_txt_lines.append(f"   - Capital en riesgo / a mover: {money(a['impact_capital'])}")
            actions_txt_lines.append(f"   - Upside economico estimado: No disponible")
            actions_txt_lines.append(f"   - Horizonte: 30-90 dias")
            actions_txt_lines.append(f"   Riesgo operativo: {a['risk']}")
    actions_txt = "\n".join(actions_txt_lines)

    # ----------------------------
    # 6.4 Residual Capital Treatment (explicit decision)
    # Preferred: withdraw if no destination above benchmark.
    # We cannot check "next subcategory with ROIC > portfolio avg" beyond what facts pack has,
    # so we state the decision deterministically and reflect economics using portfolio ROIC.
    # ----------------------------
    residual = float(residual_capital) if "residual_capital" in locals() else 0.0
    residual_treatment = []
    residual_treatment.append(f"Residual capital: {money(residual)}")
    if residual <= 0:
        residual_treatment.append("Decision: No residual capital.")
        residual_treatment.append("Expected economic impact: No disponible")
    else:
        residual_treatment.append("Decision: Retirar capital del sistema operativo mediante reduccion neta de inventario.")
        # Impact: removing capital increases ROIC mechanically if profit unchanged.
        # But we do not invent profit change; we state impact on denominator only.
        residual_treatment.append("Expected economic impact: Mejora mecanica del ROIC por reduccion de capital invertido; gross profit incremental no disponible.")
    residual_treatment_txt = "\n".join(residual_treatment)

    # ----------------------------
    # Validation Economic (explicit)
    # ----------------------------
    validation_txt = []
    validation_txt.append("incremental_profit = capital_reallocated * (destination_roic - source_roic)")
    validation_txt.append(f"capital_reallocated (assigned): {money(capital_reassigned if 'capital_reassigned' in locals() else None)}")
    validation_txt.append(f"incremental_profit (computed): {money(inc_profit_total if 'inc_profit_total' in locals() else None)}")
    validation_txt = "\n".join(validation_txt)

    # ----------------------------
    # Executive Conclusion (4–6 sentences, required phrase)
    # ----------------------------
    # Note: We do NOT change your existing numbers; we only state from computed values here.
    # If new_roic cannot be computed, we keep it "No disponible".
    concl = []
    concl.append(f"El capital de inventario esta concentrado y parcialmente mal asignado frente al benchmark interno (P75).")
    concl.append(f"El ROIC actual (proxy) es {roic_x(roic_weighted)} frente a {roic_x(p75)}; gap {roic_x(gap_vs_p75)}.")
    concl.append(f"El upside proxy total es {money(upside)} equivalente a {pct(upside_pct)} del capital en alcance.")
    concl.append(f"El plan propone liberar {money(capital_released if 'capital_released' in locals() else None)} y reasignar {money(capital_reassigned if 'capital_reassigned' in locals() else None)} hacia subcategorias con ROIC superior.")
    concl.append(f"El efecto economico esperado es un incremento de gross profit de {money(inc_profit_total if 'inc_profit_total' in locals() else None)} y un nuevo ROIC estimado de {roic_x(new_roic) if new_roic is not None else 'No disponible'}.")
    concl.append("La mejora proviene exclusivamente de reasignacion de capital existente.")
    concl_txt = " ".join(concl)

    # ----------------------------
    # IC Decision Statement (Section 9)
    # ----------------------------
    decision_txt = "\n".join([
        "Decision: Approve capital reallocation plan as proposed.",
        f"Expected economic result: ROIC increase from {roic_x(roic_weighted)} to {roic_x(new_roic) if new_roic is not None else 'No disponible'}",
        "Execution owner: Inventory leadership",
        "Execution timeline: 90 days",
        "Objective: Maximize portfolio-level ROIC",
    ])

    # ----------------------------
    # Compose memo (strict, no emojis)
    # ----------------------------
    memo = f"""
INVENTORY CAPITAL MEMO
Empresa: {company_name}
Snapshot: {snapshot_date}
Preparado por: {prepared_by}

1. Scope
- SKUs en alcance (capital > 0): {safe_str(skus_in_scope)}
- Categorias en alcance: {safe_str(categories_in_scope)}
- Subcategorias en alcance: {safe_str(subcategories_in_scope)}
- Capital de inventario en alcance: {money(total_capital)}
Nota: No incluye CxC/CxP; es una lectura economica del capital en inventario.

2. Current Performance (Inventory Capital Proxy)
- Gross profit 12m estimado: {money(gp_current)}
- Inventory ROIC (capital-weighted): {roic_x(roic_weighted)}

3. Internal Benchmark
- ROIC P75: {roic_x(p75)}
- Gap vs P75: {roic_x(gap_vs_p75)}

4. Upside Potential (Inventory Only, proxy)
- Gross profit at P75: {money(gp_at_p75)}
- Incremental upside vs current: {money(upside)}
- Upside / inventory capital: {pct(upside_pct)}

5. Capital & Upside Concentration
5.1 Por Category (ordenado por inventory_capital)
{cat_table}

5.2 Por Subcategory (top por capital)
{sub_cap_table}

5.3 Por Subcategory (top por upside)
{sub_up_table}

6. Capital Reallocation Plan (30-90 dias)
Plan status: {safe_str(plan_note)}

6.1 Capital to Reduce
{reduce_table}

6.2 Capital to Increase
{increase_table}

6.3 Expected Portfolio Effect
- Capital liberado: {money(capital_released if 'capital_released' in locals() else None)}
- Capital reasignado: {money(capital_reassigned if 'capital_reassigned' in locals() else None)}
- Capital no asignado: {money(residual_capital if 'residual_capital' in locals() else None)}
- Incremental gross profit esperado: {money(inc_profit_total if 'inc_profit_total' in locals() else None)}
- Nuevo ROIC estimado del portafolio: {roic_x(new_roic) if new_roic is not None else "No disponible"}

Validacion economica:
{validation_txt}

6.4 Residual Capital Treatment
{residual_treatment_txt}

6.5 New Portfolio Allocation
{alloc_table}

7. Priority Actions (30-90 dias)
{actions_txt}

8. Appendix tecnico
- Inventory ROIC proxy = gross_profit_12m_est / inventory_capital.
- Benchmark interno P75 = percentil 75 de ROIC proxy a nivel SKU.
- Upside proxy = (inventory_capital_total * P75) - gross_profit_12m_est_total.

9. Investment Committee Decision
{decision_txt}
""".strip()

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if out_filename is None:
        # Use snapshot_date from facts pack (preferred)
        suffix = snapshot_date if snapshot_date != "No disponible" else datetime.today().strftime("%Y-%m-%d")
        out_filename = f"inventory_capital_memo_{suffix}.txt"

    out_path = out_dir / out_filename
    out_path.write_text(memo, encoding="utf-8")

    return memo, out_path


# ============================================
# RUN (adjust facts_path if needed)
# ============================================
memo, path = generate_inventory_capital_memo(
    facts_path="reports/facts_pack_2026-02-18.json",
    out_dir="reports",
    out_filename=None,
    top_n_subcats_for_plan=12,
)

print(memo)
print(f"\nSaved memo -> {path}")

INVENTORY CAPITAL MEMO
Empresa: AdventureWorks Demo
Snapshot: 2026-02-18
Preparado por: Sintelo

1. Scope
- SKUs en alcance (capital > 0): 219
- Categorias en alcance: 4
- Subcategorias en alcance: 35
- Capital de inventario en alcance: 19,225,834
Nota: No incluye CxC/CxP; es una lectura economica del capital en inventario.

2. Current Performance (Inventory Capital Proxy)
- Gross profit 12m estimado: 5,776,873
- Inventory ROIC (capital-weighted): 0.30x

3. Internal Benchmark
- ROIC P75: 0.49x
- Gap vs P75: 0.19x

4. Upside Potential (Inventory Only, proxy)
- Gross profit at P75: 9,394,162
- Incremental upside vs current: 3,617,289
- Upside / inventory capital: 18.8%

5. Capital & Upside Concentration
5.1 Por Category (ordenado por inventory_capital)
   category inventory_capital inventory_roic upside_gp upside_per_cap cap_share
      Bikes        14,623,653          0.34x 5,012,206          34.3%     76.1%
 Components         4,375,837          0.05x 1,918,187          43.8%     22.8%